In [ ]:
import json
from dataclasses import dataclass
from typing import Literal

from dotenv import  load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph,START,END
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from loguru import logger
from langchain.messages import HumanMessage,ToolMessage

import random
load_dotenv(override = True)


@tool(parse_docstring=True)
def get_weather(city:str)->str:
    """
    查询指定城市的当日天气

    Args:
        city:城市名称
    """
    return f"{city}天气晴朗,微风"

tools = [get_weather]

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

model_with_tool = model.bind_tools(tools=tools)

# 直接使用Messages作为状态
#1. 定义llm节点
def llm_node(state:MessagesState) -> MessagesState:
    messages = state["messages"]
    res = model_with_tool.invoke(messages)
    return {
        "messages":[res]
    }
#2. 定义路由 执行完llm_node之后判断是否需要进行工具调用
def router(state:MessagesState) -> Literal["tool_node",END]:
    if state["messages"][-1].tool_calls:
        return "tool_node"
    return END
global_cache = dict()
def wrap_tool_call(request,execute):
    tool_name=request.tool_call["name"]
    tool_args=json.dumps(request.tool_call["args"])
    tool_call_id=request.runtime.tool_call_id
    #1. 判断同样的函数和参数是否调用过 => 直接使用缓存
    cache_key=(tool_name,tool_args)
    cache=global_cache.get(cache_key)
    if cache:
        #缓存命中 之前有过调用
        logger.info("{}调用缓存命中",tool_name)
        tool_msg=ToolMessage(
            tool_call_id=tool_call_id,
            content=cache
        )
    else:
        #缓存未命中 需要执行工具函数
        tool_msg=execute(request)
        logger.info("将{}的调用结果写入缓存",tool_name)
        global_cache[cache_key]=tool_msg.content
    return tool_msg
#3.构建图
builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",ToolNode(tools=tools,wrap_tool_call=wrap_tool_call))
builder.add_edge(START,"llm_node")
builder.add_conditional_edges("llm_node",router,path_map=["tool_node",END])
builder.add_edge("tool_node","llm_node")

graph = builder.compile()

from IPython.display import display
display(graph)
print("=" * 30,"第一次调用 北京 ","=" * 30)
res = graph.invoke({"messages":[HumanMessage(content="今天北京的天气如何?")]})
print(res)
for msg in res["messages"]:
    msg.pretty_print()

In [ ]:
print("=" * 30,"第二次调用 北京 ","=" * 30)
res2 = graph.invoke({"messages":[HumanMessage(content="今天北京的天气如何?")]})
print(res2)
for msg in res2["messages"]:
    msg.pretty_print()